# Classification Pipeline: Titanic Survival Prediction

**Dataset:** `titanic.csv`

**Objective:** Build an end-to-end data engineering pipeline using `ColumnTransformer` and `Pipeline` to predict passenger survival.

**Key Concepts:**

- **ColumnTransformer:** Applies different preprocessing steps to different feature types (numeric vs categorical).
- **Pipeline:** Chains preprocessing and model training into a single object, preventing data leakage.
- **Serialization:** Saving the entire pipeline for future use.

---


### Step 1: Setup & Data Loading


In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv('../data/raw/titanic.csv')

print(f"Dataset Shape: {df.shape}")
df.head()

### Step 2: Define the Preprocessing Pipeline

We drop irrelevant columns and define specific steps for numerical and categorical features.


In [ ]:
# Define features and target
X = df.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df['Survived']

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

# Numeric Pipeline: Median Imputation + Scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Mode Imputation + One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline defined.")

### Step 3: Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")

### Step 4: Build & Train the Full Pipeline

We attach a `RandomForestClassifier` to our preprocessor.


In [ ]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the model
full_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = full_pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))